Load Result File 
===

In [ ]:
import matplotlib.pyplot as plt
from ansys.dpf import core as dpf
from pathlib import Path
import numpy as np
from numpy.linalg import norm
from utils import HIC_calculation


In [ ]:

project_root = Path.cwd().resolve().parent if Path.cwd().name == "scripts" else Path.cwd().resolve()
dyna_folder_path = project_root / "DYNA_files"
dyna_path = dyna_folder_path / "head_impact_test"
d3plot = str(dyna_path / "d3plot")
ds = dpf.DataSources()
ds.set_result_file_path(d3plot, "d3plot")
model = dpf.Model(ds)
print(model)

### Extract Data Fields (Time + Acceleration)

In [ ]:
# Get total time and all acceleration
time = np.array(model.metadata.time_freq_support.time_frequencies.data)/1000 # second unit
acceleration_field_container:dpf.FieldsContainer = model.results.acceleration.on_all_time_freqs.eval() # mm/ms^2 unit

# Get acceleration of head nodes over_time
node_id = [3607962] ## TODO: node ID to get acceleration from ##
acceleration = []
for time_ind in range(len(time)):
    # average (ax,ay,az) of nodes in head (ax,ay,az)
    acc_head_nodes = np.zeros([len(node_id), 3])
    for (id_ind, id) in enumerate(node_id):
        acc_head_nodes[id_ind] = acceleration_field_container[time_ind].get_entity_data_by_id(id).squeeze()
    acc_head_nodes_avg = np.average(acc_head_nodes, axis=0)
    
    # resultant acc
    acc_avg = norm(acc_head_nodes_avg) # mm/ms^2 unit
    acc_avg = acc_avg/(9.8E-3) # gravity unit
    acceleration.append(acc_avg) 

acc = np.array(acceleration) # convert to np


### Calculate HIC

In [ ]:
# Select collision duration
start_time = 0
end_time = 0.03
select_duration_index = np.where((time < end_time) & (time > start_time))
HIC_duration = 15  # in milliseconds

# Calculate HIC
max_HIC, time_at_max_HIC, index_at_max_HIC, end_index = HIC_calculation(
    time[select_duration_index], 
    acc[select_duration_index], 
    HIC_duration
)

print(f'Max HIC: {max_HIC:.2f} at time: {time_at_max_HIC:.4f} s')
print(f'Total HIC time duration: {time[end_index]-time[index_at_max_HIC]:.4f} s')

### Plot Acceleration/Time and HIC region

In [ ]:
# Plot figure
plt.figure(figsize=(6, 4))
plt.rcParams.update({
    "font.size":12,  # for labels, ticks, legend, etc.
})
time_draw = time*1000
plt.plot(time_draw[select_duration_index],acc[select_duration_index], '-', linewidth=2)
# Highlight the HIC region
plt.axvspan(time_draw[index_at_max_HIC], time_draw[end_index], color='red', alpha=0.3, label='HIC Duration')
plt.legend()
plt.xlabel('time (ms)')
plt.ylabel('acceleration (g)')
plt.title('Acceleration of head over time')
plt.grid()
plt.show()